# 04 — Aprendizado não-supervisionado

**Objetivo:** agrupar aeroportos por perfil operacional/atraso e visualizar
essa estrutura em 2D usando PCA.

**Por que aeroportos (e não voos)?** Cluster de voos individuais é ruidoso e
pouco interpretável. Cluster de aeroportos responde a uma pergunta concreta
do desafio: *"É possível agrupar aeroportos com perfis semelhantes?"*

**Abordagens:**
1. **K-Means** sobre features agregadas por aeroporto (perfil de atraso e operação).
2. **PCA** para reduzir a dimensionalidade a 2 componentes e visualizar.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from src import config
from src.data_loader import load_airports, load_flights
from src.preprocessing import clean_pipeline
from src.features import feature_pipeline
from src.models.clustering import (
    build_airport_features,
    cluster_airports,
    cluster_profiles,
    find_optimal_k,
    reduce_pca,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Construir features por aeroporto

In [ ]:
flights = load_flights(sample_size=config.SAMPLE_SIZE)
flights = clean_pipeline(flights)
flights = feature_pipeline(flights)

feats = build_airport_features(flights, min_flights=5000)
print(f"Aeroportos retidos (≥5k voos): {len(feats)}")
feats.describe().round(3)

In [ ]:
feats.head()

## 2. Escolher K com elbow + silhouette

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(feats.values)

elbow = find_optimal_k(X, k_range=range(2, 10))
elbow

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(elbow["k"], elbow["inertia"], marker="o")
axes[0].set_title("Inércia (elbow)")
axes[0].set_xlabel("K")
axes[1].plot(elbow["k"], elbow["silhouette"], marker="o", color="orange")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("K")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "14_kmeans_selection.png", dpi=120)
plt.show()

## 3. Rodar K-Means com o K escolhido

In [ ]:
k = int(elbow.loc[elbow["silhouette"].idxmax(), "k"])
print(f"K escolhido (silhouette máxima): {k}")

clustered, km, scaler = cluster_airports(feats, n_clusters=k, random_state=config.RANDOM_SEED)
clustered["cluster"].value_counts().sort_index()

## 4. Perfil médio de cada cluster — interpretação

In [ ]:
profiles = cluster_profiles(clustered)
profiles

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    profiles.drop(columns=["n_flights", "n_destinations", "n_airlines"], errors="ignore"),
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    ax=ax,
)
ax.set_title("Perfil médio por cluster (features padronizadas)")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "15_cluster_profiles.png", dpi=120)
plt.show()

## 5. Visualização 2D — PCA

In [ ]:
X_scaled = StandardScaler().fit_transform(feats.values)
X_2d, pca = reduce_pca(X_scaled, n_components=2, random_state=config.RANDOM_SEED)
print("Variância explicada por componente:", pca.explained_variance_ratio_.round(3))
print("Acumulada:", pca.explained_variance_ratio_.sum().round(3))

In [ ]:
plot_df = pd.DataFrame(X_2d, columns=["PC1", "PC2"], index=feats.index)
plot_df["cluster"] = clustered["cluster"].values
plot_df["delay_rate"] = feats["delay_rate"].values
plot_df["n_flights"] = feats["n_flights"].values

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    plot_df["PC1"],
    plot_df["PC2"],
    c=plot_df["cluster"],
    cmap="tab10",
    s=plot_df["n_flights"] / plot_df["n_flights"].max() * 200 + 20,
    alpha=0.8,
    edgecolors="k",
    linewidths=0.5,
)
for idx, row in plot_df.iterrows():
    if row["n_flights"] > plot_df["n_flights"].quantile(0.85):
        ax.annotate(idx, (row["PC1"], row["PC2"]), fontsize=8, alpha=0.7)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Aeroportos no espaço PCA — cor=cluster, tamanho=volume de voos")
legend = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "16_pca_2d.png", dpi=120)
plt.show()

## 6. Listar aeroportos por cluster

In [ ]:
for c in sorted(clustered["cluster"].unique()):
    members = clustered[clustered["cluster"] == c]
    print(f"\n── Cluster {c} ({len(members)} aeroportos)")
    print(f"   média de atraso: {members['delay_rate'].mean():.2%}")
    print(f"   média de voos:  {members['n_flights'].mean():,.0f}")
    sample = members.sort_values("n_flights", ascending=False).head(15).index.tolist()
    print(f"   maiores: {sample}")

## 7. Componentes principais — quais features dominam cada eixo?

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feats.columns,
    columns=["PC1", "PC2"],
).round(3)
print("Loadings (correlação feature × componente):")
loadings

## 8. Persistir artefatos

In [ ]:
joblib.dump(
    {"kmeans": km, "scaler": scaler, "pca": pca, "feature_names": list(feats.columns)},
    config.MODELS_DIR / "airport_clustering.joblib",
)
clustered.to_csv(config.REPORTS_DIR / "airport_clusters.csv")
profiles.to_csv(config.REPORTS_DIR / "cluster_profiles.csv")
print("✓ artefatos salvos.")

## 9. Discussão

**O que esperar dos clusters (a confirmar após execução)**:
- **Hubs grandes com atraso alto** (ORD, EWR, SFO): muitos voos, várias companhias, taxa de atraso acima da média, peso grande de NAS/clima.
- **Hubs grandes "saudáveis"** (ATL, DTW): volume alto, mas taxa de atraso baixa — boa gestão operacional.
- **Aeroportos médios regionais**: volume moderado, taxa de atraso próxima da média, poucas companhias.
- **Aeroportos sazonais/turísticos**: distância média alta (voos longos), perfil sazonal forte.

**Limitações**:
- K-Means assume clusters esféricos — em features muito assimétricas (`n_flights` cauda longa), pode separar outliers em vez de grupos reais. Tentar DBSCAN/HDBSCAN como exploração futura.
- A escolha de features influencia o resultado; outras pode revelar padrões diferentes (ex.: variação intra-anual, tipos de aeronave).

**Próximos passos**:
- Cruzar clusters com geografia (lat/lon de `airports.csv`) para ver se há padrões regionais.
- Clusterizar também pares (origem, mês) para detectar problemas sazonais por aeroporto.
- Usar os clusters como feature em modelos supervisionados (e.g. "esse voo sai de um hub-problema").